# EAGLE Draft Model Training for Gemma 4 E2B

Trains an EAGLE-style draft model for speculative decoding on CoreML ANE.

- **Architecture**: FC fusion + SwiGLU FFN (~33M params)
- **Input**: target hidden state + token embedding
- **Output**: hidden state for shared lm_head
- **Target**: 1.5-2x speedup (31 -> 45-60 tok/s) with K=3 draft tokens

**Runtime**: T4 ~1h, A100 ~20min

After training, convert with `build_speculative.py` and deploy on iPhone.

In [ ]:
!pip install -q -U transformers accelerate datasets tqdm safetensors huggingface_hub hf_transfer

import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"  # faster downloads

import transformers
print(f"transformers {transformers.__version__}")

# Paste your token here (https://huggingface.co/settings/tokens)
HF_TOKEN = ""  # <-- paste between quotes, e.g. "hf_abc..."

assert HF_TOKEN, "Paste your HuggingFace token above"
from huggingface_hub import login
login(token=HF_TOKEN)
print("Logged in.")

In [ ]:
import gc
import json
import math
import os
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

# ---- Config ----
MODEL_ID = "google/gemma-4-E2B-it"
HIDDEN_SIZE = 1536
FF_SIZE = 6144        # SwiGLU FFN width
SOFTCAP = 30.0
EMBED_SCALE = HIDDEN_SIZE ** 0.5  # 39.19

# Data
NUM_SAMPLES = 5000    # sequences to collect
SEQ_LEN = 512         # tokens per sequence

# Training
EPOCHS = 10
BATCH_SIZE = 256
LR = 3e-4
WARMUP_STEPS = 500
FEATURE_LOSS_W = 0.0  # set >0 to add MSE(draft_hidden, target_hidden)
K_DRAFT = 3           # draft steps for eval

DEVICE = "cuda"
SAVE_DIR = "/content/eagle_draft"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Device: {DEVICE}, GPU: {torch.cuda.get_device_name()}")

In [ ]:
# ============================================================
# Draft model architecture
# ============================================================

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        norm = torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() * norm).to(x.dtype) * self.weight


class DraftModel(nn.Module):
    """EAGLE-style draft model.

    Approximates the target model's hidden state transition:
        draft(h_t, embed(tok_{t+1})) -> h_{t+1}

    During inference, autoregressive drafting:
        step 0: draft(target_h, embed(target_tok)) -> d_h0, tok_1
        step 1: draft(d_h0, embed(tok_1))          -> d_h1, tok_2
        step 2: draft(d_h1, embed(tok_2))          -> d_h2, tok_3
        verify [tok_1, tok_2, tok_3] with target model in batch.
    """

    def __init__(self, hidden_size=1536, ff_size=6144):
        super().__init__()
        self.hidden_size = hidden_size
        # Fuse target hidden + token embedding
        self.fc = nn.Linear(hidden_size * 2, hidden_size, bias=False)
        # SwiGLU FFN block
        self.norm1 = RMSNorm(hidden_size)
        self.gate_proj = nn.Linear(hidden_size, ff_size, bias=False)
        self.up_proj = nn.Linear(hidden_size, ff_size, bias=False)
        self.down_proj = nn.Linear(ff_size, hidden_size, bias=False)
        self.norm2 = RMSNorm(hidden_size)
        self._init_weights()

    def _init_weights(self):
        for m in [self.fc, self.gate_proj, self.up_proj, self.down_proj]:
            nn.init.xavier_uniform_(m.weight)

    def forward(self, target_hidden, token_embed):
        # target_hidden: (B, hidden)  token_embed: (B, hidden)
        x = self.fc(torch.cat([target_hidden, token_embed], dim=-1))
        residual = x
        x = self.norm1(x)
        x = self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))
        x = residual + x
        x = self.norm2(x)
        return x  # (B, hidden)


draft = DraftModel(HIDDEN_SIZE, FF_SIZE).to(DEVICE)
n_params = sum(p.numel() for p in draft.parameters())
print(f"Draft model: {n_params:,} params ({n_params * 4 / 1e6:.1f} MB fp32)")

In [ ]:
# ============================================================
# Load target model + collect training data
# ============================================================
from transformers import Gemma4ForConditionalGeneration, AutoTokenizer

print(f"Loading {MODEL_ID}...")
target = Gemma4ForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map=DEVICE, token=HF_TOKEN,
)
target.eval()
for p in target.parameters():
    p.requires_grad = False

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
embed_fn = target.get_input_embeddings()
lm_head_weight = target.lm_head.weight.data.clone().cpu().half()
print(f"  hidden={HIDDEN_SIZE}  vocab={lm_head_weight.shape[0]}  lm_head={lm_head_weight.shape}")

In [ ]:
# ============================================================
# Collect hidden states + embeddings from diverse text
# ============================================================
from datasets import load_dataset

ds = load_dataset("wikitext", "wikitext-103-raw-v1", split="train")

all_h_in = []    # hidden[t]       -- input hidden state
all_e_in = []    # embed(tok[t+1]) -- next-token embedding (EAGLE convention)
all_h_tgt = []   # hidden[t+1]     -- feature target
all_tok_tgt = [] # argmax(lm_head(hidden[t+1])) -- token target
collected = 0

print(f"Collecting {NUM_SAMPLES} samples, seq_len={SEQ_LEN}...")
pbar = tqdm(total=NUM_SAMPLES)

with torch.no_grad():
    for row in ds:
        if collected >= NUM_SAMPLES:
            break
        text = row["text"].strip()
        if len(text) < 200:
            continue

        ids = tokenizer.encode(text, return_tensors="pt",
                               truncation=True, max_length=SEQ_LEN).to(DEVICE)
        N = ids.shape[1]
        if N < 32:
            continue

        # Hidden states (with final RMSNorm applied by HF model)
        out = target.model(input_ids=ids, output_hidden_states=False)
        hidden = out.last_hidden_state[0].cpu().half()  # (N, 1536)

        # Scaled token embeddings
        embeds = embed_fn(ids)[0].cpu().half() * EMBED_SCALE  # (N, 1536)

        # EAGLE training pairs:
        #   input:  (hidden[t],   embed(tok[t+1]))  for t=0..N-2
        #   target: hidden[t+1],  argmax(lm_head(hidden[t+1]))
        all_h_in.append(hidden[:-1])    # (N-1, 1536)
        all_e_in.append(embeds[1:])     # (N-1, 1536) -- embed of next token
        all_h_tgt.append(hidden[1:])    # (N-1, 1536)

        # Self-distillation: target model's argmax at each position
        logits = F.linear(hidden[1:].float(), lm_head_weight.float())
        all_tok_tgt.append(logits.argmax(dim=-1))  # (N-1,)

        collected += 1
        pbar.update(1)
        if collected % 500 == 0:
            total_tok = sum(h.shape[0] for h in all_h_in)
            print(f"  {collected} samples, {total_tok:,} token pairs")

pbar.close()
total_tok = sum(h.shape[0] for h in all_h_in)
print(f"Done: {collected} samples, {total_tok:,} token pairs")

In [ ]:
# ============================================================
# Free target model, build DataLoader
# ============================================================
del target, embed_fn, ds
gc.collect()
torch.cuda.empty_cache()
print("Target model freed.")

# Concatenate all pairs
h_in = torch.cat(all_h_in, dim=0)      # (M, 1536)
e_in = torch.cat(all_e_in, dim=0)      # (M, 1536)
h_tgt = torch.cat(all_h_tgt, dim=0)    # (M, 1536)
tok_tgt = torch.cat(all_tok_tgt, dim=0) # (M,)
del all_h_in, all_e_in, all_h_tgt, all_tok_tgt
gc.collect()

M = h_in.shape[0]
print(f"Total pairs: {M:,}  Memory: {(h_in.nbytes + e_in.nbytes + h_tgt.nbytes) / 1e9:.2f} GB")

# Train / test split (95/5)
split = int(M * 0.95)
train_ds = TensorDataset(h_in[:split], e_in[:split], h_tgt[:split], tok_tgt[:split])
test_ds  = TensorDataset(h_in[split:], e_in[split:], h_tgt[split:], tok_tgt[split:])
del h_in, e_in, h_tgt, tok_tgt
gc.collect()

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          pin_memory=True, drop_last=True, num_workers=2)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
                          pin_memory=True, num_workers=2)

print(f"Train: {len(train_ds):,} pairs ({len(train_loader)} batches)")
print(f"Test:  {len(test_ds):,} pairs")

# Move lm_head to GPU (frozen, fp16)
lm_head_w = lm_head_weight.to(DEVICE)  # (vocab, 1536) fp16
print(f"lm_head on GPU: {lm_head_w.nbytes / 1e6:.0f} MB")

In [ ]:
# ============================================================
# Training loop
# ============================================================
optimizer = torch.optim.AdamW(draft.parameters(), lr=LR, weight_decay=0.01)
total_steps = EPOCHS * len(train_loader)

def lr_schedule(step):
    if step < WARMUP_STEPS:
        return step / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_schedule)
scaler = torch.amp.GradScaler("cuda")

print(f"Training: {EPOCHS} epochs, {total_steps} steps, lr={LR}")
print(f"Feature loss weight: {FEATURE_LOSS_W}")

best_loss = float("inf")
step = 0

for epoch in range(EPOCHS):
    draft.train()
    ep_loss = 0
    ep_tok_loss = 0
    ep_feat_loss = 0
    t0 = time.time()

    for h, e, ht, tt in train_loader:
        h = h.to(DEVICE, non_blocking=True)
        e = e.to(DEVICE, non_blocking=True)
        ht = ht.to(DEVICE, non_blocking=True)
        tt = tt.to(DEVICE, non_blocking=True)

        with torch.amp.autocast("cuda", dtype=torch.float16):
            out = draft(h, e)  # (B, 1536) fp16
            # Token loss: CE against target model's argmax
            logits = F.linear(out, lm_head_w)  # (B, vocab) fp16
            token_loss = F.cross_entropy(logits.float(), tt)

            loss = token_loss
            if FEATURE_LOSS_W > 0:
                feat_loss = F.mse_loss(out.float(), ht.float())
                loss = loss + FEATURE_LOSS_W * feat_loss
            else:
                feat_loss = torch.tensor(0.0)

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(draft.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        ep_loss += loss.item()
        ep_tok_loss += token_loss.item()
        ep_feat_loss += feat_loss.item()
        step += 1

    n = len(train_loader)
    dt = time.time() - t0

    # Validation
    draft.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for h, e, ht, tt in test_loader:
            h, e, tt = h.to(DEVICE), e.to(DEVICE), tt.to(DEVICE)
            with torch.amp.autocast("cuda", dtype=torch.float16):
                out = draft(h, e)
                logits = F.linear(out, lm_head_w)
            val_loss += F.cross_entropy(logits.float(), tt).item()
            val_correct += (logits.argmax(-1) == tt).sum().item()
            val_total += tt.shape[0]

    vl = val_loss / len(test_loader)
    acc = val_correct / val_total

    print(f"E{epoch+1:02d}  train={ep_tok_loss/n:.4f}  "
          f"feat={ep_feat_loss/n:.4f}  "
          f"val={vl:.4f}  acc={acc:.1%}  "
          f"lr={scheduler.get_last_lr()[0]:.2e}  {dt:.0f}s")

    # Save best
    if vl < best_loss:
        best_loss = vl
        torch.save(draft.state_dict(), f"{SAVE_DIR}/eagle_draft_best.pt")
        print(f"  -> saved best (val={vl:.4f})")

# Save final
torch.save(draft.state_dict(), f"{SAVE_DIR}/eagle_draft_final.pt")
print(f"\nTraining complete. Best val loss: {best_loss:.4f}")

In [ ]:
# ============================================================
# Evaluate: simulate autoregressive drafting
# ============================================================
print(f"\nEvaluating acceptance rate (K={K_DRAFT} draft steps)...")

draft.eval()
draft.load_state_dict(torch.load(f"{SAVE_DIR}/eagle_draft_best.pt", weights_only=True))

# Reload embed weights for autoregressive sim
from transformers import Gemma4ForConditionalGeneration

print("Loading target embeddings for eval...")
target_eval = Gemma4ForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map=DEVICE, token=HF_TOKEN,
)
target_eval.eval()
embed_fn_eval = target_eval.get_input_embeddings()

accepted = [0] * K_DRAFT
total_eval = 0
EVAL_CHUNK = 512

test_h = test_ds.tensors[0].to(DEVICE)
test_e = test_ds.tensors[1].to(DEVICE)
test_ht = test_ds.tensors[2].to(DEVICE)
test_tt = test_ds.tensors[3].to(DEVICE)

max_eval = test_h.shape[0] - K_DRAFT

with torch.no_grad():
    for start in tqdm(range(0, max_eval, EVAL_CHUNK)):
        end = min(start + EVAL_CHUNK, max_eval)
        B = end - start

        h = test_h[start:end]
        e = test_e[start:end]

        for k in range(K_DRAFT):
            with torch.amp.autocast("cuda", dtype=torch.float16):
                draft_h = draft(h, e)
                logits = F.linear(draft_h, lm_head_w)
                logits = torch.tanh(logits / SOFTCAP) * SOFTCAP
                pred = logits.argmax(-1)

            gt = test_tt[start + k : end + k]
            accepted[k] += (pred == gt).sum().item()

            h = draft_h
            e = embed_fn_eval(pred.unsqueeze(0)).squeeze(0).half() * EMBED_SCALE

        total_eval += B

print(f"\nResults ({total_eval:,} positions):")
for k in range(K_DRAFT):
    rate = accepted[k] / total_eval
    print(f"  Step {k}: acceptance = {rate:.1%}")

avg_tok = 1 + sum(accepted[k] / total_eval for k in range(K_DRAFT))
print(f"\nExpected tokens/round: {avg_tok:.2f}")
draft_overhead = K_DRAFT * 0.03
speedup = avg_tok / (1 + draft_overhead)
print(f"Estimated speedup: {speedup:.2f}x (31 -> {31 * speedup:.0f} tok/s)")

del target_eval, embed_fn_eval
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# ============================================================
# Save config + weights for CoreML conversion
# ============================================================

config = {
    "architecture": "eagle_ffn",
    "hidden_size": HIDDEN_SIZE,
    "ff_size": FF_SIZE,
    "softcap": SOFTCAP,
    "embed_scale": EMBED_SCALE,
    "trainable_params": sum(p.numel() for p in draft.parameters()),
    "training": {
        "model_id": MODEL_ID,
        "num_samples": NUM_SAMPLES,
        "seq_len": SEQ_LEN,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "feature_loss_weight": FEATURE_LOSS_W,
        "best_val_loss": best_loss,
    },
}
with open(f"{SAVE_DIR}/eagle_config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"Saved to {SAVE_DIR}/")
print(f"  eagle_draft_best.pt   ({os.path.getsize(f'{SAVE_DIR}/eagle_draft_best.pt') / 1e6:.1f} MB)")
print(f"  eagle_draft_final.pt  ({os.path.getsize(f'{SAVE_DIR}/eagle_draft_final.pt') / 1e6:.1f} MB)")
print(f"  eagle_config.json")
print()
print("Next steps:")
print("  1. Download eagle_draft_best.pt + eagle_config.json")
print("  2. python build_speculative.py --eagle-path ./eagle_draft/eagle_draft_best.pt")
print("  3. Deploy eagle.mlpackage to iPhone")

In [ ]:
# ============================================================
# (Optional) Copy to Google Drive
# ============================================================
try:
    from google.colab import drive
    drive.mount("/content/drive")
    import shutil
    dst = "/content/drive/MyDrive/eagle_draft"
    os.makedirs(dst, exist_ok=True)
    for f in ["eagle_draft_best.pt", "eagle_draft_final.pt", "eagle_config.json"]:
        shutil.copy2(f"{SAVE_DIR}/{f}", f"{dst}/{f}")
    print(f"Copied to {dst}/")
except Exception as ex:
    print(f"Drive mount skipped: {ex}")